# Extract letter texts

Extract the annotation version of the letters from the local TEI files. The original volume files supply the footnotes; the letter-only XML files supply the letter divisions.

In [ ]:
from shared import paths
from IPython.display import display


## Extraction

In [ ]:
paths.prepare_directories()
from lxml import etree
import os
import json
from os import listdir, path
import pickle
import xml.etree.ElementTree as ET
import re


def remove_newlines_and_spaces(text):
    text = re.sub('\\n', '', text)
    text = re.sub('\\s+', ' ', text)
    return text


def fn_lookup(id, file):
    try:
        with open(str(paths.XML_DIR) + '/' + file) as file_with_fn:
            fn_xml = file_with_fn.read()
            fn_root = etree.fromstring(fn_xml)
            footnote = ''.join(
                fn_root.xpath(
                    f".//note[@type='footnote' and @xml:id='{id}']",
                    namespaces={'xml': 'http://www.w3.org/XML/1998/namespace'},
                )[0].itertext(),
            ).strip()
            output_note = remove_newlines_and_spaces(footnote)
            output_note += '\n'
        return output_note
    except IndexError:
        print(f'IndexError occurred for id: {id}')
        return None


def count_letter_tags(xml_file):
    try:
        tree = ET.parse(xml_file)
        root = tree.getroot()
        letter_count = len(root.findall(".//div[@type='letter']"))
        return letter_count
    except ET.ParseError:
        print(f'Error parsing XML file: {xml_file}')
        return None


def find_values_starting_with_S(xml_elements):
    values_starting_with_S = []
    for element in xml_elements:
        rend_attribute = element.get('rend')
        if rend_attribute and rend_attribute.startswith('S'):
            values_starting_with_S.append(rend_attribute)
    if values_starting_with_S:
        return values_starting_with_S
    else:
        return None
indent = [
    'indent_16em nopad',
    'indent_8em nopad',
    'indent_16em',
    'indent_18em nopad',
    'indent_14em',
    'indent_2em nopad',
    'indent_10',
    'indent_20em',
    'indent_10em',
    'indent_2em',
    'indent_12em',
    'indent_6em',
    'indent_6em nopad',
    'indent_20em nopad',
    'indent_10em nopad',
    'indent_4em',
    'indent_8em',
    'indent_14em nopad',
    'indent_12em nopad',
    'indent_4em nopad',
    'tbindent',
    'tbindent indent_2em',
    'tbindenttb',
    'indent_12em nopad',
    'indent_6em nopad',
]
files = [f for f in listdir(os.getcwd()) if f.endswith('.xml')]
ext = '.txt'
whole_letters = {}
for file in files:
    let_count = count_letter_tags(paths.LETTERS_XML_DIR / file)
    with open(paths.LETTERS_XML_DIR / file) as Letters:
        xml = Letters.read()
        root = etree.fromstring(xml)
        years = root.findall(".//div[@type='year']")
        page_number = '1'
        for elem_y in years:
            year = elem_y.find('head')
            months = elem_y.findall(".//div[@type='month']")
            per_month_dic = {
                'January': [],
                'February': [],
                'March': [],
                'April': [],
                'May': [],
                'June': [],
                'July': [],
                'August': [],
                'September': [],
                'October': [],
                'November': [],
                'December': [],
                '?': [],
            }
            for elem_m in months:
                month = elem_m.find('head')
                letters = elem_m.findall("div[@type='letter']")
                for elem_l in letters:
                    np_after_letter = []
                    letter_dic = {}
                    content = {}
                    letter_dic['xml_id'] = elem_l.xpath('./@xml:id')[0]
                    all_tags = elem_l.getchildren()
                    elements_end = all_tags[-1].getchildren()
                    if elements_end:
                        if elements_end[-1].tag == 'pb':
                            np_after_letter.append(elements_end[-1].xpath('./@n')[0])
                    page_number_list = []
                    np_within_letter = elem_l.xpath('.//pb/@n')
                    if np_within_letter:
                        for np in np_within_letter:
                            if np in np_after_letter:
                                if not page_number_list:
                                    page_number_list.append(page_number)
                                    page_number = np
                                else:
                                    page_number = np
                                    break
                            else:
                                page_number_list.append(page_number)
                                page_number = np
                                page_number_list.append(page_number)
                    else:
                        page_number_list.append(page_number)
                    page_number_list = list(set(page_number_list))
                    page_number_list.sort()
                    letter_dic['pages'] = page_number_list
                    head_string = ''
                    information_string = ''
                    letter_string = ''
                    paragraph_list = []
                    fn_collect = []
                    number = 1
                    exclude_list = ['right', 'block']
                    for big_tag in all_tags:
                        values_starting_with_S = find_values_starting_with_S(all_tags)
                        first_sig = values_starting_with_S[0] if values_starting_with_S else None
                        rend_value = big_tag.get('rend')
                        if big_tag.tag == 'p':
                            if rend_value not in exclude_list:
                                if values_starting_with_S:
                                    if rend_value not in values_starting_with_S:
                                        if rend_value:
                                            paragraph_list.append(big_tag.xpath('string(.)'))
                            if rend_value in indent or rend_value == 'right':
                                letter_string += '\n' + ' ' * 5 if rend_value in indent else ' ' * 50
                        for element in big_tag.iter():
                            if element.text:
                                if element.tag == 'emph':
                                    emphasized = '**' + element.text + '**'
                                    content = remove_newlines_and_spaces(emphasized)
                                else:
                                    content = remove_newlines_and_spaces(element.text)
                            else:
                                content = ''
                            if element.tail:
                                if element.tag == 'ptr':
                                    if not element.get('n')[1:].isalpha():
                                        content = '[[{0}]]'.format(number)
                                        number += 1
                                        target_value = element.get('target')
                                        fn_collect.append(fn_lookup(target_value, file))
                                        content += remove_newlines_and_spaces(element.tail)
                                    else:
                                        content += remove_newlines_and_spaces(element.tail)
                                else:
                                    content += remove_newlines_and_spaces(element.tail)
                            if big_tag.tag == 'head':
                                head_string += content
                            elif big_tag.tag == 'p':
                                if rend_value == 'block':
                                    information_string += content
                                elif big_tag.get('rend') == first_sig:
                                    content = '\n' + content
                                    letter_string += content
                                else:
                                    letter_string += content
                            else:
                                letter_string += content
                        if big_tag.tag == 'p':
                            letter_string += '\n'
                    fn_list = '\n'.join([f'[[{i + 1}]] \n{fn}' for i, fn in enumerate(fn_collect)])
                    concatenated_text = '\n\n'.join(
                        [
                            head_string,
                            information_string,
                            '_' * 60,
                            letter_string,
                            '_' * 60,
                            fn_list,
                        ],
                    )
                    letter_dic['content'] = concatenated_text
                    letter_dic['paragraphs'] = paragraph_list
                    per_month_dic[''.join(month.text.split())].append(letter_dic)
            print('done with {0}'.format(''.join(year.text.split())))
            whole_letters[''.join(year.text.split())] = per_month_dic
file_path = paths.ANNOTATION_EXPORT
with open(file_path, 'wb') as f:
    pickle.dump(whole_letters, f)
print('Dictionary stored successfully.')
count = 0
for year in whole_letters:
    for month in whole_letters[year]:
        if month:
            for letter in whole_letters[year][month]:
                count += 1
print(count)
